# 03b — Análise Geográfica

Investigamos como o desempenho varia entre os **27 estados** brasileiros e as **5 regiões**, incluindo um mapa interativo gerado a partir do GeoJSON oficial do IBGE.

**Seções:**
1. Nota média por estado (ranking)
2. Análise por região geográfica
3. Heatmap: estado × área de conhecimento
4. Mapa interativo (choropleth)
5. Principais observações

In [ ]:
import sys
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import plotly.express as px

sys.path.insert(0, '../src')
from utils import NOTAS_COLS, NOTAS_LABELS, configurar_estilo, salvar_figura

configurar_estilo()

df = pd.read_parquet('../data/processed/amostra_limpa.parquet')
print(f'Shape: {df.shape}')

# Mapeamento estado → região
REGIOES = {
    'Norte':        ['AM', 'PA', 'AC', 'RO', 'RR', 'AP', 'TO'],
    'Nordeste':     ['MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA'],
    'Centro-Oeste': ['MT', 'MS', 'GO', 'DF'],
    'Sudeste':      ['SP', 'RJ', 'MG', 'ES'],
    'Sul':          ['PR', 'SC', 'RS'],
}
UF_REGIAO = {uf: reg for reg, ufs in REGIOES.items() for uf in ufs}
df['REGIAO'] = df['SG_UF_PROVA'].map(UF_REGIAO)

# O GeoJSON do IBGE usa código numérico (codarea), não a sigla.
# Este mapeamento converte codarea → sigla para o choropleth.
IBGE_SIGLA = {
    '11': 'RO', '12': 'AC', '13': 'AM', '14': 'RR', '15': 'PA', '16': 'AP', '17': 'TO',
    '21': 'MA', '22': 'PI', '23': 'CE', '24': 'RN', '25': 'PB', '26': 'PE',
    '27': 'AL', '28': 'SE', '29': 'BA',
    '31': 'MG', '32': 'ES', '33': 'RJ', '35': 'SP',
    '41': 'PR', '42': 'SC', '43': 'RS',
    '50': 'MS', '51': 'MT', '52': 'GO', '53': 'DF',
}

## 1. Ranking de Estados por Nota Média

In [ ]:
nota_uf = (
    df.groupby('SG_UF_PROVA', observed=True)
    .agg(nota_media=('NOTA_MEDIA', 'mean'), n=('NOTA_MEDIA', 'count'))
    .round(1)
    .sort_values('nota_media')
)
nota_uf['REGIAO'] = nota_uf.index.map(UF_REGIAO)

CORES_REGIAO = {
    'Norte': '#1f77b4', 'Nordeste': '#ff7f0e',
    'Centro-Oeste': '#2ca02c', 'Sudeste': '#d62728', 'Sul': '#9467bd'
}

media_geral = nota_uf['nota_media'].mean()

fig, ax = plt.subplots(figsize=(10, 11))
bars = ax.barh(
    nota_uf.index, nota_uf['nota_media'],
    color=nota_uf['REGIAO'].map(CORES_REGIAO), alpha=0.85
)
for bar, val in zip(bars, nota_uf['nota_media']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', fontsize=8)

ax.axvline(media_geral, color='black', linestyle='--', linewidth=1)

legendas = [Patch(facecolor=cor, label=reg) for reg, cor in CORES_REGIAO.items()]
legendas.append(plt.Line2D([0], [0], color='black', linestyle='--',
                            label=f'Média: {media_geral:.1f}'))
ax.legend(handles=legendas, loc='lower right', fontsize=8)

ax.set_xlabel('Nota Média')
ax.set_title('Ranking de Estados por Nota Média — ENEM 2023', fontsize=13)
ax.set_xlim(nota_uf['nota_media'].min() - 5, nota_uf['nota_media'].max() + 12)
plt.tight_layout()
salvar_figura(fig, '03b_ranking_estados')
plt.show()

variacao = nota_uf['nota_media'].max() - nota_uf['nota_media'].min()
print(f'Variação entre melhor e pior estado: {variacao:.1f} pts')
print(f'Melhor: {nota_uf.index[-1]} ({nota_uf["nota_media"].iloc[-1]:.1f})')
print(f'Pior:   {nota_uf.index[0]}  ({nota_uf["nota_media"].iloc[0]:.1f})')

## 2. Análise por Região Geográfica

In [ ]:
ordem_regioes = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
paleta = [CORES_REGIAO[r] for r in ordem_regioes]

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(
    data=df, x='REGIAO', y='NOTA_MEDIA',
    order=ordem_regioes, palette=paleta,
    width=0.5, flierprops=dict(marker='o', markersize=1.5, alpha=0.2), ax=ax
)
medianas_reg = df.groupby('REGIAO', observed=True)['NOTA_MEDIA'].median()
for i, reg in enumerate(ordem_regioes):
    if reg in medianas_reg.index:
        ax.text(i, medianas_reg[reg] + 6, f'{medianas_reg[reg]:.0f}',
                ha='center', fontsize=9, fontweight='bold')

ax.set_title('Distribuição da Nota Média por Região — ENEM 2023', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Nota Média')
plt.tight_layout()
salvar_figura(fig, '03b_nota_por_regiao')
plt.show()

In [ ]:
medias_reg = (
    df.groupby('REGIAO', observed=True)[NOTAS_COLS]
    .mean()
    .reindex(ordem_regioes)
    .round(1)
)
medias_reg.columns = [NOTAS_LABELS[c] for c in medias_reg.columns]
medias_reg

## 3. Heatmap: Estado × Área de Conhecimento

In [ ]:
pivot_uf = (
    df.groupby('SG_UF_PROVA', observed=True)[NOTAS_COLS]
    .mean()
    .round(1)
)
pivot_uf.columns = [NOTAS_LABELS[c] for c in pivot_uf.columns]
pivot_uf = pivot_uf.sort_values('Matemática')

fig, ax = plt.subplots(figsize=(11, 13))
sns.heatmap(
    pivot_uf, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.3, ax=ax, cbar_kws={'label': 'Nota Média'},
    annot_kws={'size': 8}
)
ax.set_title('Nota Média por Estado e Área de Conhecimento — ENEM 2023', fontsize=13)
ax.set_ylabel('Estado')
ax.set_xlabel('')
plt.tight_layout()
salvar_figura(fig, '03b_heatmap_uf_area')
plt.show()

## 4. Mapa Interativo

Choropleth interativo com o GeoJSON oficial do IBGE. O IBGE retorna apenas o código numérico (`codarea`) por feature — o mapeamento `IBGE_SIGLA` definido na célula de imports converte para a sigla do estado.

Passe o mouse sobre cada estado para ver nota média, região e número de candidatos.

In [ ]:
URL_IBGE = (
    'https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR'
    '?formato=application/vnd.geo+json&qualidade=minima&intrarregiao=UF'
)

try:
    resp = requests.get(URL_IBGE, timeout=20)
    resp.raise_for_status()
    geojson_br = resp.json()

    # Enriquece cada feature com a sigla, usando IBGE_SIGLA como lookup
    for feat in geojson_br['features']:
        codarea = feat['properties']['codarea']
        feat['properties']['sigla'] = IBGE_SIGLA.get(codarea, codarea)

    print(f'GeoJSON carregado: {len(geojson_br["features"])} estados')
    print('Exemplo:', geojson_br['features'][0]['properties'])

except Exception as e:
    geojson_br = None
    print(f'Erro ao carregar GeoJSON: {e}')

In [ ]:
if geojson_br is None:
    print('GeoJSON indisponível — verifique a conexão e rode novamente.')
else:
    df_mapa = (
        nota_uf.reset_index()
        .rename(columns={'SG_UF_PROVA': 'UF', 'nota_media': 'Nota Média',
                         'n': 'Candidatos', 'REGIAO': 'Região'})
    )

    fig = px.choropleth_mapbox(
        df_mapa,
        geojson=geojson_br,
        locations='UF',
        featureidkey='properties.sigla',
        color='Nota Média',
        color_continuous_scale='YlOrRd',
        mapbox_style='white-bg',
        zoom=3.2,
        center={'lat': -14, 'lon': -52},
        opacity=0.85,
        hover_data={'UF': True, 'Nota Média': ':.1f', 'Região': True, 'Candidatos': ':,'},
        labels={'Nota Média': 'Nota Média'},
        title='Nota Média por Estado — ENEM 2023',
    )
    fig.update_layout(
        margin={'r': 0, 't': 40, 'l': 0, 'b': 0},
        paper_bgcolor='white',
        coloraxis_colorbar=dict(title='Nota Média'),
    )

    config = {'scrollZoom': True}
    fig.write_html('../reports/figures/03b_mapa_interativo.html', config=config)
    print('Mapa salvo em reports/figures/03b_mapa_interativo.html')
    fig.show(config=config)

## 5. Principais Observações

| # | Observação | Evidência |
|---|---|---|
| 1 | Variação expressiva entre estados | 67 pts entre MG (572,7) e AM (505,6) |
| 2 | Sul e Sudeste lideram com folga | Mediana Sul/Sudeste ~557–561 pts vs Norte ~509 pts (+52 pts) |
| 3 | Matemática é a área com maior desigualdade regional | Amplitude de 87 pts entre MG (565,7) e AP (478,9) |
| 4 | DF se destaca dentro do Centro-Oeste | 558,3 pts — 16,7 pts acima da média nacional (541,6) |

**Próximo passo:** `04_insights_finais.ipynb` — storytelling com os principais achados do projeto.